In [1]:
import os
import sys
from pathlib import Path

# Asegura que el notebook encuentre los módulos del proyecto
# (útil si Jupyter fue lanzado desde otra carpeta)
PROJECT_DIR = Path(os.getcwd())

print(PROJECT_DIR)

c:\Users\rmend\Dropbox\Personal_Desktop2026\BootCamp ML\DMC\Diplomado_AI_Engineering\DesignImplementation_Chatbots\week5\ReportFinanciaFraudChatBox


In [2]:
import os
import sys
from pathlib import Path

# Asegura que el notebook encuentre los módulos del proyecto

if not (PROJECT_DIR / "vector_search.py").exists():
    
    # find directory's parents until we find the project root
    
    for parent in [PROJECT_DIR, *PROJECT_DIR.parents]:
        if (parent / "vector_search.py").exists():
            PROJECT_DIR = parent
            break

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print("Directorio del proyecto:", PROJECT_DIR)


Directorio del proyecto: c:\Users\rmend\Dropbox\Personal_Desktop2026\BootCamp ML\DMC\Diplomado_AI_Engineering\DesignImplementation_Chatbots\week5\ReportFinanciaFraudChatBox


In [3]:
from config import Config

Config.print_config()

# Verificación rápida de dependencias
import chromadb
import numpy

print(f"✓ chromadb {chromadb.__version__} | numpy {numpy.__version__} | Python {sys.version.split()[0]}")



SESIÓN 6 - RECUPERACIÓN DE INFORMACIÓN (RAG)
Base de datos vectorial: ChromaDB (persistente)
Directorio de persistencia: ./chroma_db
Colección: politicas_empresa

Proveedor de embeddings: sentence-transformers
Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dimensión: 384

Chunking: size=500, overlap=50
Búsqueda: top_k=5, score_threshold=0.0
RAG: 3 chunks de contexto en el prompt

✓ chromadb 1.5.9 | numpy 2.4.6 | Python 3.11.1


---
## Parte 1 — Preprocesamiento y Chunking *(Taller Práctico Parte 1)*

> 📄 **Extracción y limpieza** + ✂️ **División en chunks**
>
> - Tamaño óptimo: 200–500 tokens por chunk
> - Overlap: 10–20 % para mantener el contexto
> - Metadata: agregar fuente, página y fecha a cada chunk

### 1.1 División de un documento largo en chunks

In [4]:
from embeddings import TextChunker

# Documento largo de ejemplo (sección de un manual técnico)
manual_tecnico = (
    "MANUAL TÉCNICO DE SEGURIDAD. Capítulo 1: Acceso a sistemas. Todo colaborador "
    "debe usar autenticación de dos factores para acceder a los sistemas internos. "
    "Las credenciales son personales e intransferibles. Capítulo 2: Respaldos. "
    "Los respaldos de la información se realizan diariamente a las tres de la "
    "mañana y se conservan copias por treinta días. Capítulo 3: Incidentes. "
    "Ante un incidente de seguridad se debe notificar al área de TI dentro de "
    "la primera hora y aislar el equipo afectado de la red. Capítulo 4: Software. "
    "Solo se permite instalar software aprobado por el área de TI. "
)

# Chunking: size y overlap configurables (recomendado 10-20% de overlap)
chunker = TextChunker(chunk_size=250,
                      chunk_overlap=50)

chunks = chunker.split_text(manual_tecnico)

print(f"Texto original: {len(manual_tecnico)} caracteres")
print(f"Chunks generados: {len(chunks)}\n")

for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ({len(chunk)} caracteres) ---")
    print(chunk[:180] + ("..." if len(chunk) > 180 else ""))
    print()


Texto original: 585 caracteres
Chunks generados: 4

--- Chunk 1 (228 caracteres) ---
MANUAL TÉCNICO DE SEGURIDAD. Capítulo 1: Acceso a sistemas. Todo colaborador debe usar autenticación de dos factores para acceder a los sistemas internos. Las credenciales son pers...

--- Chunk 2 (194 caracteres) ---
rsonales e intransferibles. Capítulo 2: Respaldos. Los respaldos de la información se realizan diariamente a las tres de la mañana y se conservan copias por treinta días. Capítulo ...

--- Chunk 3 (200 caracteres) ---
n copias por treinta días. Capítulo 3: Incidentes. Ante un incidente de seguridad se debe notificar al área de TI dentro de la primera hora y aislar el equipo afectado de la red. C...

--- Chunk 4 (112 caracteres) ---
l equipo afectado de la red. Capítulo 4: Software. Solo se permite instalar software aprobado por el área de TI.



### 1.2 Extracción de texto de los PDFs de la empresa

La carpeta `pdfs/` contiene documentos de ejemplo (catálogo de productos y un
documento técnico). El lector extrae y limpia el texto de cada archivo.

In [5]:
from pdf_reader import PDFReader

# Extrae y limpia el texto de todos los PDFs de la carpeta
reader = PDFReader()
documentos_pdf = reader.read_pdf_folder("./pdfs")

print("\nVista previa del primer documento:")
print(documentos_pdf[0]["metadata"]["filename"])
print(documentos_pdf[0]["text"][:300], "...")


Encontrados 4 archivos PDF
  Leyendo: 1244218-ley-29571_spij.pdf... ✓ (266440 caracteres)
  Leyendo: 504-2021.R.pdf... ✓ (66996 caracteres)
  Leyendo: CATÁLOGO DE PRODUCTOS 2025.pdf... ✓ (26508 caracteres)
  Leyendo: INCYTU_18-012.pdf... ✓ (26765 caracteres)

✓ Total documentos procesados: 4

Vista previa del primer documento:
1244218-ley-29571_spij.pdf
CÓDIGO DE PROTECCIÓN Y DEFENSA DEL CONSUMIDOR LEY Nº 29571 (VERSIÓN DEL CÓDIGO DE PROTECCIÓN Y DEFENSA DEL CONSUMIDOR EN INGLÉS - MAYO 2015) Promulgado : 01-09-2010. Publicado : 02-09-2010. (*) De conformidad con la , publicada el 02 septiembre 2010, el presente Código entra en vigencia a los treint ...


---
## Parte 2 — Embeddings y Similitud Coseno *(Taller Práctico Parte 2)*

> 🧮 **Vectorización de textos**
>
> - `SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')` → vectores de 384 dimensiones
> - Textos similares → vectores cercanos en el espacio
>
> 📊 **Similitud coseno:** `cos(θ) = (A · B) / (||A|| × ||B||)`
> - 0.9–1.0 → Muy similar | 0.7–0.9 → Similar | < 0.7 → Poco similar

### 2.1 Convertir textos en vectores (embeddings)

In [6]:
from embeddings import EmbeddingModel

# Carga el modelo configurado en .env
# (la primera vez lo descarga de HuggingFace; ya descargado, funciona offline)
embedder = EmbeddingModel()

info = embedder.get_model_info()
print(f"Modelo: {info['model_name']}")
print(f"Dimensión de los vectores: {info['dimension']}\n")

# Como en la diapositiva: "gato -> [0.8, 0.99, 0.1, ...]"
textos_ejemplo = [
    "El gato duerme sobre el sofá",
    "El perro duerme en el patio",
    "El avión despega a las seis de la mañana",
    "La programación orientada a objetos usa clases y herencia",
]

embeddings_ejemplo = embedder.embed_documents(textos_ejemplo, show_progress=False)

for texto, vector in zip(textos_ejemplo, embeddings_ejemplo):
    print(f"'{texto[:40]}...' -> [{', '.join(f'{v:.2f}' for v in vector[:8])}, ...]")


Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Modelo cargado. Dimensión: 384
Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dimensión de los vectores: 384

'El gato duerme sobre el sofá...' -> [0.42, -0.25, -0.15, 0.22, 0.09, 0.31, -0.01, 0.24, ...]
'El perro duerme en el patio...' -> [0.34, 0.01, -0.38, 0.13, 0.45, 0.15, -0.33, 0.01, ...]
'El avión despega a las seis de la mañana...' -> [0.43, 0.20, -0.24, 0.16, -0.00, -0.06, 0.15, 0.24, ...]
'La programación orientada a objetos usa ...' -> [-0.14, 0.21, -0.21, 0.03, -0.15, -0.09, 0.05, -0.08, ...]


### 2.2 Calcular similitud coseno con la fórmula de la sesión

In [7]:
import numpy as np
from embeddings import cosine_similarity, cosine_similarity_matrix

# Vectores del ejemplo anterior
a = embeddings_ejemplo[0]   # gato
b = embeddings_ejemplo[1]   # perro
c = embeddings_ejemplo[3]   # programación

# Fórmula de la sesión: cos(θ) = (A · B) / (||A|| × ||B||)
dot_producto = np.dot(a, b)
normas = np.linalg.norm(a) * np.linalg.norm(b)
sim_manual = dot_producto / normas

print("PASO A PASO con la fórmula:")
print(f"  A · B         = {dot_producto:.4f}")
print(f"  ||A|| × ||B|| = {normas:.4f}")
print(f"  cos(θ)        = {dot_producto:.4f} / {normas:.4f} = {sim_manual:.4f}")
print(f"  (verificación con la función del proyecto: {cosine_similarity(a, b):.4f})")

# Similitudes interesantes
print(f"\n  sim(gato, perro)         = {cosine_similarity(a, b):.4f}  → muy similar")
print(f"  sim(gato, programación)   = {cosine_similarity(a, c):.4f}  → poco similar")

# Matriz de similitud entre todos los pares
etiquetas = ["gato", "perro", "avión", "programación"]
matriz = cosine_similarity_matrix(embeddings_ejemplo)

print("\nMatriz de similitud coseno (simétrica, diagonal = 1):")
print("          " + "  ".join(f"{e:>12}" for e in etiquetas))
for nombre, fila in zip(etiquetas, matriz):
    print(f"{nombre:>9}  " + "  ".join(f"{v:>12.3f}" for v in fila))


PASO A PASO con la fórmula:
  A · B         = 8.4068
  ||A|| × ||B|| = 22.1143
  cos(θ)        = 8.4068 / 22.1143 = 0.3801
  (verificación con la función del proyecto: 0.3801)

  sim(gato, perro)         = 0.3801  → muy similar
  sim(gato, programación)   = 0.0121  → poco similar

Matriz de similitud coseno (simétrica, diagonal = 1):
                  gato         perro         avión  programación
     gato         1.000         0.380         0.016         0.012
    perro         0.380         1.000        -0.025        -0.012
    avión         0.016        -0.025         1.000        -0.014
programación         0.012        -0.012        -0.014         1.000


---
## Parte 3 — Indexación y Búsqueda Semántica con ChromaDB *(Taller Práctico Parte 3)*

> 🗂️ **Indexación con ChromaDB** (base de datos vectorial persistente, sin Docker)
>
> - `client.create_collection()` → crea la colección
> - `collection.add(documents, embeddings, ids)` → agrega documentos
> - Persistencia: los vectores se guardan en disco (`./chroma_db`)

### 3.1 API directa de ChromaDB (tal como en la diapositiva)

In [8]:
import chromadb

# 1. Cliente persistente: los vectores se guardan en ./chroma_db
client = chromadb.PersistentClient(path="./chroma_db")

# 2. Crear colección usando similitud coseno
coleccion = client.get_or_create_collection(
    name="demo_notebook",
    embedding_function=None,            # usamos los embeddings de nuestro modelo
    metadata={"hnsw:space": "cosine"},  # métrica: coseno
)

# 3. Agregar documentos: collection.add(documents, embeddings, ids)
documentos_demo = [
    "El teletrabajo está permitido dos días por semana",
    "Las vacaciones son quince días hábiles por año",
    "El horario laboral es de 9:00 a 18:00",
    "Los gastos de viaje se reembolsan en quince días",
]
ids_demo = [f"demo_{i}" for i in range(len(documentos_demo))]
embeddings_demo = embedder.embed_documents(documentos_demo, show_progress=False)

# upsert: si el ID ya existe lo actualiza (permite re-ejecutar la celda)
coleccion.upsert(ids=ids_demo, documents=documentos_demo, embeddings=embeddings_demo)

print(f"✓ {coleccion.count()} documentos en la colección '{coleccion.name}'")


✓ 4 documentos en la colección 'demo_notebook'


In [9]:
len(embeddings_demo)

4

### 3.2 Búsqueda semántica: `collection.query(...)` con Top-K

In [10]:
# 4. Consulta: se vectoriza la pregunta y se buscan los K más similares
pregunta = "¿Cuántos días de vacaciones me corresponden?"
query_embedding = embedder.embed_query(pregunta)

K = 2
resultado = coleccion.query(
    query_embeddings=[query_embedding],
    n_results=K,
    include=["documents", "distances"],
)


# include is what the user wants to see in the output.
# In this case, we want to see the documents and their distances from the query.

print(f"Pregunta: '{pregunta}'\n")
for doc, distance in zip(resultado["documents"][0], resultado["distances"][0]):
    score = 1 - distance  # distance = 1 - coseno  →  score = similitud coseno
    print(f"  [score={score:.3f}] {doc}")


Pregunta: '¿Cuántos días de vacaciones me corresponden?'

  [score=0.647] Las vacaciones son quince días hábiles por año
  [score=0.474] El teletrabajo está permitido dos días por semana


### 3.3 Motor completo: `VectorSearchEngine` (caso de uso de la sesión)

Indexa la base de políticas internas de la empresa y responde por **significado**,
aunque la pregunta no contenga las mismas palabras que el documento.

In [11]:
from vector_search import VectorSearchEngine

# Caso de uso: políticas internas, manuales y documentación legal
engine = VectorSearchEngine(collection_name="politicas_empresa")

# collection related to Políticas internas, manuales y documentación legal

# create collection (delete_if_exists=True) to ensure a clean state for the example

engine.create_collection(delete_if_exists=True)

politicas = [
    "Las políticas internas permiten el teletrabajo dos días por semana con autorización del jefe directo",
    "El manual técnico establece que las contraseñas deben cambiarse cada noventa días",
    "La documentación legal exige firmar un acuerdo de confidencialidad al ingresar a la empresa",
    "El reglamento interno prohíbe el uso de dispositivos personales para datos confidenciales",
    "La política de capacitación otorga veinte horas anuales de formación pagadas por la empresa",
    "El plan de emergencia indica evacuar por las escaleras en caso de sismo",
    "Los gastos de viaje se reembolsan en un plazo máximo de quince días",
    "El código de ética prohíbe aceptar regalos de proveedores",
]
metadatas = [{"fuente": f"documento_{i//4 + 1}", "seccion": i % 4} for i in range(len(politicas))]
ids_politicas = [f"pol_{i}" for i in range(len(politicas))]

# Indexar: los embeddings se generan automáticamente
n = engine.add_documents(politicas, metadatas=metadatas, ids=ids_politicas, show_progress=False)
print(f"\n✓ {n} documentos indexados en '{engine.collection_name}'")


Inicializando motor de búsqueda semántica (ChromaDB)...
Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Modelo cargado. Dimensión: 384
ChromaDB persistente en: ./chroma_db
Motor de búsqueda inicializado

✓ Colección 'politicas_empresa' eliminada
✓ Colección 'politicas_empresa' lista (0 documentos existentes)
Indexando 8 documentos...
Generando embeddings...
✓ 8 documentos indexados en 'politicas_empresa'

✓ 8 documentos indexados en 'politicas_empresa'


In [12]:
# Búsqueda por significado: ninguna pregunta repite las palabras exactas
consultas = [
    "¿Puedo trabajar desde mi casa?",
    "¿Qué dice la empresa sobre las contraseñas?",
    "¿Qué debo hacer si tiembla?",
    "¿Me pagan los cursos de formación?",
]

for pregunta in consultas:
    print(f"\n{'=' * 70}")
    print(f"Pregunta: '{pregunta}'")
    print('=' * 70)


    # search 2 top chunks  
     
    resultados = engine.search(pregunta, k=2)

    for i, r in enumerate(resultados, 1):
        print(f"{i}. [score={r['score']:.3f}] (fuente: {r['metadata'].get('fuente')})")
        print(f"   {r['text']}")



Pregunta: '¿Puedo trabajar desde mi casa?'
1. [score=0.307] (fuente: documento_1)
   Las políticas internas permiten el teletrabajo dos días por semana con autorización del jefe directo
2. [score=0.174] (fuente: documento_2)
   El plan de emergencia indica evacuar por las escaleras en caso de sismo

Pregunta: '¿Qué dice la empresa sobre las contraseñas?'
1. [score=0.563] (fuente: documento_1)
   La documentación legal exige firmar un acuerdo de confidencialidad al ingresar a la empresa
2. [score=0.545] (fuente: documento_1)
   El manual técnico establece que las contraseñas deben cambiarse cada noventa días

Pregunta: '¿Qué debo hacer si tiembla?'
1. [score=0.147] (fuente: documento_2)
   El plan de emergencia indica evacuar por las escaleras en caso de sismo

Pregunta: '¿Me pagan los cursos de formación?'
1. [score=0.531] (fuente: documento_2)
   La política de capacitación otorga veinte horas anuales de formación pagadas por la empresa
2. [score=0.207] (fuente: documento_2)
   Los g

### 3.4 Filtros por metadatos (`where=...`)

Combinar búsqueda vectorial con metadatos, como en la diapositiva:
`where={'fuente': 'manual'}`.

In [13]:
# Filtrar resultados por metadata: solo respuestas de un documento
pregunta = "¿Qué normas de seguridad existen?"
resultados_filtrados = engine.search(
    pregunta,
    k=3,
    filter_metadata={"fuente": "documento_1"},   # solo chunks de ese documento
)

# output search filtered by Documento 1  


print(f"Pregunta: '{pregunta}' — filtro fuente=documento_1\n")
for i, r in enumerate(resultados_filtrados, 1):
    print(f"{i}. [score={r['score']:.3f}] {r['text']}")

# Estadísticas de la colección
print("\nEstadísticas:")
for clave, valor in engine.get_stats().items():
    print(f"  {clave}: {valor}")


Pregunta: '¿Qué normas de seguridad existen?' — filtro fuente=documento_1

1. [score=0.457] El reglamento interno prohíbe el uso de dispositivos personales para datos confidenciales
2. [score=0.404] La documentación legal exige firmar un acuerdo de confidencialidad al ingresar a la empresa
3. [score=0.252] Las políticas internas permiten el teletrabajo dos días por semana con autorización del jefe directo

Estadísticas:
  document_count: 8
  collection_name: politicas_empresa
  persist_dir: ./chroma_db
  embedding_dimension: 384
  model_name: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  exists: True


### 3.5 Indexar los PDFs reales (chunking + embeddings + ChromaDB)

In [ ]:
# Si la celda de lectura de PDFs no se ejecutó en esta sesión, se re-lee
try:
    documentos_pdf
except NameError:
    from pdf_reader import PDFReader
    documentos_pdf = PDFReader().read_pdf_folder("./pdfs")

# Colección persistente para los PDFs
engine_pdf = VectorSearchEngine(collection_name="pdfs_empresa")
engine_pdf.create_collection(delete_if_exists=True)

# Chunking de los PDFs (son largos) + indexación
n = engine_pdf.add_documents_chunked(
    [doc["text"] for doc in documentos_pdf],
    chunk_size=500,      # 200-500 recomendado en la sesión
    chunk_overlap=50,    # 10% de overlap
    metadatas=[doc["metadata"] for doc in documentos_pdf],
    show_progress=True,
)
print(f"\n✓ {n} chunks indexados en '{engine_pdf.collection_name}' (persistidos en ./chroma_db)")


Inicializando motor de búsqueda semántica (ChromaDB)...
Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Modelo cargado. Dimensión: 384
ChromaDB persistente en: ./chroma_db
Motor de búsqueda inicializado

✓ Colección 'pdfs_empresa' lista (0 documentos existentes)
Dividiendo 4 documentos en chunks...


Procesando documentos:   0%|          | 0/4 [03:27<?, ?it/s]


In [ ]:
# Buscar en los PDFs: cambia las consultas y vuelve a ejecutar esta celda
# (no hace falta reindexar: la colección ya está persistida)
consultas_pdf = [
    "¿Cuáles son los productos del catálogo?",
    "¿Qué es la inteligencia artificial según el documento?",
]

for pregunta in consultas_pdf:
    print(f"\n{'=' * 70}")
    print(f"Pregunta: '{pregunta}'")
    print('=' * 70)

    resultados = engine_pdf.search(pregunta, k=3)

    for i, r in enumerate(resultados, 1):
        fuente = r["metadata"].get("filename", "desconocida")
        print(f"{i}. [score={r['score']:.3f}] 📄 {fuente}")
        print(f"   {r['text'][:200]}...")



Pregunta: '¿Cuáles son los productos del catálogo?'
1. [score=0.439] 📄 CATÁLOGO DE PRODUCTOS 2025.pdf
   ction es nuestra gama de clásicos del sabor belga. Consiste en chocolates perfectamente equilibrados aclamados por su versatilidad y garantizando excelentes resultados en todos los usos de chocolate y...
2. [score=0.413] 📄 CATÁLOGO DE PRODUCTOS 2025.pdf
   un balance perfecto entre dulzura, cacao y leche. 100% man- teca de cacao y 100% vainilla natural. Chispas de chocolate negro para hor- near, ideales para galletas, muffins, paste- les, etc. Hecho con...
3. [score=0.376] 📄 CATÁLOGO DE PRODUCTOS 2025.pdf
   abor a fresa para decorar productos de pastelería. Nuestra gama de brillos de alta calidad, Miroir hace los bavarois y Mousses brillantes y atractivos. Miroir es el producto perfecto para reflejar est...

Pregunta: '¿Qué es la inteligencia artificial según el documento?'
1. [score=0.703] 📄 INCYTU_18-012.pdf
   pto de inteligencia per se no es del todo preciso. En términos coloq

---
## Bonus — RAG (Retrieval Augmented Generation)

> 🔄 **Fase de consulta online de la arquitectura RAG:**
>
> 1. Usuario envía pregunta → 2. Embedding → 3. Top-K chunks más similares
> → 4. Prompt: instrucción + chunks recuperados + pregunta → 5. LLM responde

### 4.1 Retrieval: recuperar contexto y armar el prompt aumentado

In [14]:
# 1) Retrieval: busca los chunks más relevantes
# 2) Augmented: los inyecta como CONTEXTO en el prompt
pregunta_rag = "¿Qué normas debo cumplir como empleado?"

prompt, contexto = engine.build_rag_prompt(pregunta_rag, k=3)

print("CONTEXTO RECUPERADO (los 3 chunks más similares):")
for i, chunk in enumerate(contexto, 1):
    print(f"  {i}. [{chunk['score']:.3f}] {chunk['text'][:100]}...")

print(f"\n{'─' * 70}")
print("PROMPT AUMENTADO (lo que recibiría el LLM):")
print('─' * 70)
print(prompt)


CONTEXTO RECUPERADO (los 3 chunks más similares):
  1. [0.357] La documentación legal exige firmar un acuerdo de confidencialidad al ingresar a la empresa...
  2. [0.340] La política de capacitación otorga veinte horas anuales de formación pagadas por la empresa...
  3. [0.340] Las políticas internas permiten el teletrabajo dos días por semana con autorización del jefe directo...

──────────────────────────────────────────────────────────────────────
PROMPT AUMENTADO (lo que recibiría el LLM):
──────────────────────────────────────────────────────────────────────
Eres un asistente corporativo. Responde ÚNICAMENTE con base en el contexto proporcionado. Si la información no está en el contexto, indícalo claramente. Cita la fuente de cada afirmación.

CONTEXTO:
[Contexto 1 - fuente: desconocida]
La documentación legal exige firmar un acuerdo de confidencialidad al ingresar a la empresa

[Contexto 2 - fuente: desconocida]
La política de capacitación otorga veinte horas anuales de formación

### 4.2 Generation: respuesta del LLM basada en los chunks

> ⚠️ **Opcional:** si en `.env` no hay `OPENAI_API_KEY`, esta celda solo muestra
> el prompt aumentado (sin costo). Con API key configurada, llama a
> `gpt-4o-mini` para generar la respuesta final.

In [15]:
# Generation: el LLM responde ÚNICAMENTE con base en el contexto recuperado
resultado_rag = engine.generate_answer(pregunta_rag, k=3)

if resultado_rag["answer"]:
    print("🤖 RESPUESTA GENERADA:")
    print(resultado_rag["answer"])
    print("\nFuentes usadas:")
    for fuente in resultado_rag["sources"]:
        print(f"  - {fuente}")
else:
    print("(Sin API key de OpenAI: se muestra solo el prompt aumentado)")
    print(resultado_rag["prompt"][:900])


(Sin API key de OpenAI: se muestra solo el prompt aumentado)
Eres un asistente corporativo. Responde ÚNICAMENTE con base en el contexto proporcionado. Si la información no está en el contexto, indícalo claramente. Cita la fuente de cada afirmación.

CONTEXTO:
[Contexto 1 - fuente: desconocida]
La documentación legal exige firmar un acuerdo de confidencialidad al ingresar a la empresa

[Contexto 2 - fuente: desconocida]
La política de capacitación otorga veinte horas anuales de formación pagadas por la empresa

[Contexto 3 - fuente: desconocida]
Las políticas internas permiten el teletrabajo dos días por semana con autorización del jefe directo

PREGUNTA: ¿Qué normas debo cumplir como empleado?

RESPUESTA:


---
## ✅ Resumen de la sesión

| Concepto | Dónde se practicó |
|---|---|
| **Embeddings** | Parte 2.1: texto → vector de 384 dimensiones |
| **Similitud coseno** | Parte 2.2: fórmula `(A·B)/(‖A‖·‖B‖)` paso a paso |
| **Chunking** | Parte 1.1: división con overlap 10–20 % |
| **Base de datos vectorial** | Parte 3: ChromaDB persistente en `./chroma_db` |
| **Búsqueda semántica** | Partes 3.2–3.5: Top-K por significado, con filtros |
| **RAG** | Parte 4: Retrieval → prompt aumentado → Generation |

**Reto:** ejecuta `python demo_presentacion.py` (o las celdas equivalentes) para
indexar la presentación de esta sesión y hacerle preguntas sobre embeddings,
RAG y bases de datos vectoriales.
